Task 01

In [1]:
import pandas as pd
import re

# 1. Load Yelp reviews dataset directly from GitHub using pandas
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/yelp.csv"
df = pd.read_csv(url)

# 2. Select text column, drop missing values, and take first 50 rows
data = df['text'].dropna().head(50).reset_index(drop=True)

# 3. Define text cleaning function
def clean_text(text):
    # Remove URLs and web links
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    # Remove user handles (@username)
    text = re.sub(r'@\w+', '', text)
    # Strip hashtag symbols while keeping the text word
    text = re.sub(r'#(\w+)', r'\1', text)
    # Convert all characters to lowercase
    text = text.lower()
    # Clean extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply cleaning function
cleaned_reviews = data.apply(clean_text)

# Display sample output
print("=== TASK 1: CLEANING OUTPUT SAMPLE ===")
for i in range(3):
    print(f"\n--- Original Review {i+1} ---")
    print(data[i][:150] + "...")
    print(f"--- Cleaned Review {i+1} ---")
    print(cleaned_reviews[i][:150] + "...")

=== TASK 1: CLEANING OUTPUT SAMPLE ===

--- Original Review 1 ---
My wife took me here on my birthday for breakfast and it was excellent.  The weather was perfect which made sitting outside overlooking their grounds ...
--- Cleaned Review 1 ---
my wife took me here on my birthday for breakfast and it was excellent. the weather was perfect which made sitting outside overlooking their grounds a...

--- Original Review 2 ---
I have no idea why some people give bad reviews about this place. It goes to show you, you can please everyone. They are probably griping about someth...
--- Cleaned Review 2 ---
i have no idea why some people give bad reviews about this place. it goes to show you, you can please everyone. they are probably griping about someth...

--- Original Review 3 ---
love the gyro plate. Rice is so good and I also dig their candy selection :)...
--- Cleaned Review 3 ---
love the gyro plate. rice is so good and i also dig their candy selection :)...


Task 02

In [2]:
import spacy
import pandas as pd

# Load spaCy small English model
nlp = spacy.load("en_core_web_sm")

# Process the first cleaned review for illustration
doc = nlp(cleaned_reviews[0])

# Prepare annotations list
annotations = []
for token in doc:
    # Get entity label if token is part of a named entity
    ent_label = token.ent_type_ if token.ent_type_ else "O"
    annotations.append({
        "Token": token.text,
        "POS Tag": token.pos_,
        "Detailed Tag": token.tag_,
        "Entity Label": ent_label
    })

# Convert to DataFrame summary table
annotation_df = pd.DataFrame(annotations)

print("=== TASK 2: LINGUISTIC ANNOTATION TABLE (FIRST REVIEW) ===")
print(annotation_df.head(25))

# Extract and display all Named Entities found across first 10 cleaned reviews
print("\n=== EXTRACTED NAMED ENTITIES ACROSS SAMPLE REVIEWS ===")
for idx, text in enumerate(cleaned_reviews[:10]):
    review_doc = nlp(text)
    entities = [(ent.text, ent.label_) for ent in review_doc.ents]
    if entities:
        print(f"Review {idx+1} Entities: {entities}")

=== TASK 2: LINGUISTIC ANNOTATION TABLE (FIRST REVIEW) ===
          Token POS Tag Detailed Tag Entity Label
0            my    PRON         PRP$            O
1          wife    NOUN           NN            O
2          took    VERB          VBD            O
3            me    PRON          PRP            O
4          here     ADV           RB            O
5            on     ADP           IN            O
6            my    PRON         PRP$            O
7      birthday    NOUN           NN            O
8           for     ADP           IN            O
9     breakfast    NOUN           NN            O
10          and   CCONJ           CC            O
11           it    PRON          PRP            O
12          was     AUX          VBD            O
13    excellent     ADJ           JJ            O
14            .   PUNCT            .            O
15          the     DET           DT            O
16      weather    NOUN           NN            O
17          was     AUX          VBD     

Task 03

In [3]:
import nltk
from nltk import CFG
from nltk.parse import ChartParser

# 1. Define custom Context-Free Grammar (CFG)
cfg_grammar = CFG.fromstring("""
  S -> NP VP
  VP -> V NP | V NP PP | V PP
  PP -> P NP
  NP -> Det N | Det Adj N | PropN | N
  Det -> 'the' | 'a' | 'my' | 'this'
  N -> 'food' | 'service' | 'restaurant' | 'pizza' | 'place'
  Adj -> 'good' | 'great' | 'delicious' | 'bad'
  PropN -> 'john' | 'yelp'
  V -> 'liked' | 'loved' | 'ate' | 'visited'
  P -> 'at' | 'with' | 'in'
""")

parser = ChartParser(cfg_grammar)

# 2. Function to validate sentence structure against grammar
def validate_sentence(sentence):
    tokens = sentence.lower().split()
    print(f"\nTesting Sentence: '{sentence}'")
    try:
        parses = list(parser.parse(tokens))
        if len(parses) > 0:
            print("Status: VALID sentence (Matches CFG rules)")
            for tree in parses:
                print("\nParse Tree:")
                tree.pretty_print()
            return True
        else:
            print("Status: INVALID sentence (Does not match CFG rules)")
            return False
    except ValueError as e:
        print(f"Status: INVALID sentence (Contains out-of-vocabulary words: {e})")
        return False

# 3. Test grammar against valid and invalid sentences
print("=== TASK 3: CFG SYNTAX PARSING ===")
valid_sent = "the restaurant served good food"  # Adjusted to match vocabulary
valid_sent_2 = "john loved the food at this place"
invalid_sent = "food the liked restaurant place"

validate_sentence(valid_sent_2)
validate_sentence(invalid_sent)

=== TASK 3: CFG SYNTAX PARSING ===

Testing Sentence: 'john loved the food at this place'
Status: VALID sentence (Matches CFG rules)

Parse Tree:
        S                                  
   _____|____________                       
  |                  VP                    
  |      ____________|________              
  |     |        |            PP           
  |     |        |         ___|____         
  NP    |        NP       |        NP      
  |     |     ___|___     |    ____|____    
PropN   V   Det      N    P  Det        N  
  |     |    |       |    |   |         |   
 john loved the     food  at this     place


Testing Sentence: 'food the liked restaurant place'
Status: INVALID sentence (Does not match CFG rules)


False

Task 04

In [4]:
import numpy as np
import pandas as pd
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
from sklearn.metrics import cohen_kappa_score

# Download VADER lexicon for sentiment analysis
nltk.download('vader_lexicon', quiet=True)

sia = SentimentIntensityAnalyzer()

# 1. Compute compound polarity scores for Yelp reviews
compound_scores = [sia.polarity_scores(text)['compound'] for text in cleaned_reviews]

# 2. Bin compound scores into categorical labels ('negative', 'neutral', 'positive') -> Annotator A
def get_sentiment_label(score):
    if score >= 0.05:
        return 'positive'
    elif score <= -0.05:
        return 'negative'
    else:
        return 'neutral'

annotator_a = np.array([get_sentiment_label(score) for score in compound_scores])

# 3. Simulate Annotator B using a randomized permutation with fixed seed
np.random.seed(42)
# Introduce slight noise/permutation to simulate another annotator
annotator_b = annotator_a.copy()
noise_indices = np.random.choice(len(annotator_b), size=int(0.2 * len(annotator_b)), replace=False)
categories = np.array(['negative', 'neutral', 'positive'])

for idx in noise_indices:
    annotator_b[idx] = np.random.choice(categories[categories != annotator_a[idx]])

# 4. Calculate Simple Agreement & Cohen's Kappa Score
simple_agreement = np.mean(annotator_a == annotator_b)
kappa_score = cohen_kappa_score(annotator_a, annotator_b)

# Print Summary Results
print("=== TASK 4: INTER-ANNOTATOR AGREEMENT RESULTS ===")
print(f"Total Reviews Annotated : {len(cleaned_reviews)}")
print(f"Simple Agreement        : {simple_agreement * 100:.2f}% ({np.sum(annotator_a == annotator_b)} / {len(cleaned_reviews)})")
print(f"Cohen's Kappa Score    : {kappa_score:.4f}")

# Display sample annotation comparison
df_agreement = pd.DataFrame({
    'Review Text': [t[:60] + '...' for t in cleaned_reviews[:10]],
    'Compound Score': compound_scores[:10],
    'Annotator A': annotator_a[:10],
    'Annotator B': annotator_b[:10],
    'Match': annotator_a[:10] == annotator_b[:10]
})

print("\n--- SAMPLE ANNOTATION COMPARISON TABLE ---")
print(df_agreement)

=== TASK 4: INTER-ANNOTATOR AGREEMENT RESULTS ===
Total Reviews Annotated : 50
Simple Agreement        : 80.00% (40 / 50)
Cohen's Kappa Score    : 0.4318

--- SAMPLE ANNOTATION COMPARISON TABLE ---
                                         Review Text  Compound Score  \
0  my wife took me here on my birthday for breakf...          0.9950   
1  i have no idea why some people give bad review...          0.9943   
2  love the gyro plate. rice is so good and i als...          0.8988   
3  rosie, dakota, and i love chaparral dog park!!...          0.9474   
4  general manager scott petello is a good egg!!!...          0.9883   
5  quiessence is, simply put, beautiful. full win...          0.9881   
6  drop what you're doing and drive here. after i...          0.9963   
7  luckily, i didn't have to travel far to make m...          0.9544   
8  definitely come for happy hour! prices are ama...          0.9483   
9  nobuo shows his unique talents with everything...          0.4588   

  Annotat